# Measurements in Multi-Qubit Systems Workbook

This workbook describes the solutions to the problems offered in the "Measurements in Multi-Qubit Systems" kata. Since the tasks are offered as programming problems, the explanations also cover some elements of Workbench that might be non-obvious for a first-time user.

In [ ]:
from psiqdk.workbench import Qubits


## Problem 1. Distinguish four basis states

Unlike in the previous task, this time measuring the first qubit won't give us any information on the second qubit, so we need to measure both qubits by calling `read` on the entire register directly.

The method automatically returns the measurement results as integers by treating them as little-endian notation (the first qubit stores the least significant bit).

In [ ]:
def measure_basis_state(reg: Qubits) -> int:
    return reg.read()

## Problem 2. Distinguish orthogonal states using partial measurements

Since the state of the first qubit is different in these states ($\ket +$ and $\ket -$, respectively), it's sufficient to measure only the first qubit in the Pauli X basis in order to distinguish the two states. A Hadamard gate is sufficient to change the basis into the computational (Pauli Z) basis for measurement.

In [ ]:
def measure_plusminus_state(reg: Qubits) -> int:
    reg[0].had()
    return reg[0].read()

## Problem 3. State Selection Using Partial Measurements

Note that if you measure the first qubit in the computational basis, then an outcome of $0$ collapses the second qubit to the state $a \ket{0} + b \ket{1}$, while an outcome of $1$ collapses the second qubit to the state $b \ket{0} + a \ket{1}$.

Thus, if $ind=0$ and you measure $0$ or if $ind=1$ and you measure $1$, then after the measurement the second qubit will be in the desired state. On the other hand, if $ind=1$ and you measure $0$, or if $ind=0$ and you measure $1$, then the state of the second qubit after the measurement isn't what you're looking for, but you can adjust it by applying the Pauli X gate.

Therefore, you can do the mid-circuit measurement on the first qubit, and apply an $X$ gate on the second qubit conditional on the measured outcome: if the measurement outcome is *different* from $ind$, you need to apply the gate.

In [ ]:
def state_selection_partial_meas(reg: Qubits, ind: int) -> None:
    outcome = reg[0].read_async()
    with outcome != ind:
        reg[1].x()

## Problem 4. State preparation using partial measurements

While it's possible to prepare the state directly using unitary rotations, it's simpler to use post-selection for preparing it.

Initially, you prepare an equal superposition of all basis states on the first two qubits by applying the Hadamard gate to each of them, and allocate an extra qubit in the $\ket{0}$ state:

$$\frac{1}{2}(\ket{00} + \ket{01} + \ket{10} + \ket{11})\otimes\ket{0}$$

The state of the first two qubits is a superposition of the state you want to prepare and the $\ket{00}$ state that you want to discard.

Now, you want to separate the first three basis states from the last one and to store this separation in the extra qubit. For example, you can keep the state of the extra qubit $\ket{0}$ for the basis states that you want to keep, and switch it to $\ket{1}$ for the basis states that you want to discard. A $CCX$ gate can do this, with the first two qubits used as control qubits and the extra qubit as target. When the gate is applied, the state of the extra qubit will only change to $\ket{1}$ if both control qubits are in the $\ket{11}$ state, which marks exactly the state that you want to discard:

$$CCX\frac{1}{2}(\ket{00} + \ket{01} + \ket{10} + \ket{11})\otimes\ket{0} = \frac{1}{2}(\ket{00} + \ket{01} + \ket{10})\otimes\ket{0} + \frac{1}{2}\ket{11}\otimes\ket{1}$$

Finally, you measure just the auxiliary qubit; this causes a partial collapse of the system to the state defined by the measurement result:

- If the result is 0, the first two qubits collapse to a state that is a linear combination of basis states which had the extra qubit in state $\ket{0}$, that is, they end up in the target state $\frac{1}{2}(\ket{00} + \ket{01} + \ket{10})\otimes\ket{0}$.
- If the result is 1, the first two qubits collapse to a state $\ket{11}$, so your goal is not achieved. The good thing is, this only happens in 25% of the cases, and you can just reset our qubits to the $\ket{00}$ state and try again.

In [ ]:
def state_preparation_partial_meas(reg: Qubits) -> None:
    aux = Qubits(num_qubits=1, name='aux', qpu=reg.qpu)
    outcome = 1
    while outcome:
        reg.write(0)
        aux.write(0)
        reg.had()
        aux.x(cond=reg)
        outcome = aux.read()
    
    aux.release()

> Copyright (c) 2026 PsiQuantum